# 분할 archive 준비: 합성 검사와 선택적 Drive 진입점

기본 실행은 네트워크 데이터셋 없이 tiny split tar.gz를 생성·선택 추출·매칭합니다. 실제 Colab 업로드·Drive API·GPU는 미검증입니다. M0/M1 전체 완료나 train_ready를 선언하지 않습니다. 최신 로컬 커밋의 소스 ZIP을 사용하고 아래 실제 Drive 셀은 기본 비활성입니다.

소스 ZIP은 64MiB/추출 크기 128MiB 상한으로 코드만 풀며 원본 데이터 archive와 다른 입력입니다. [준비 계약](../docs/ARCHIVE_PREPARATION.md), [Colab 안내](../docs/COLAB_SETUP.md)를 함께 확인하세요.


In [ ]:
import sys, os, shutil
print({"python":sys.version.split()[0],"free_disk_bytes":shutil.disk_usage(".").free})


In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import io, stat, tempfile, zipfile
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("소스 ZIP 한 개를 선택하세요.")
zip_name, zip_bytes = next(iter(uploaded.items()))
if not zip_name.endswith(".zip") or len(zip_bytes) > 64*1024**2:
    raise ValueError("64MiB 이하의 소스 ZIP 필요")
extract_root = Path(tempfile.mkdtemp(prefix="rewind_source_", dir="/content"))
excluded = {".git", ".venv", "data", "outputs", "checkpoints", ".aws", ".codex", ".agents"}
with zipfile.ZipFile(io.BytesIO(zip_bytes)) as archive:
    members = archive.infolist()
    if sum(m.file_size for m in members) > 128*1024**2:
        raise ValueError("소스 ZIP의 해제 크기 상한 초과")
    for member in members:
        p = PurePosixPath(member.filename)
        if p.is_absolute() or ".." in p.parts or any(":" in part for part in p.parts) or stat.S_ISLNK(member.external_attr >> 16):
            raise ValueError("잘못된 ZIP member 경로")
        if not any(part in excluded for part in p.parts):
            archive.extract(member, extract_root)
roots = [p.parent.parent for p in extract_root.rglob("rewind_archive/__init__.py")]
if len(roots) != 1:
    raise ValueError("rewind_archive 패키지가 하나인 프로젝트 ZIP 필요")
PROJECT_ROOT = roots[0]
print("source:", PROJECT_ROOT)
stamp = PROJECT_ROOT / "SOURCE_REVISION.txt"
print("revision:", stamp.read_text().strip() if stamp.exists() else "unversioned; code hash로 기록")
os.chdir(PROJECT_ROOT)
sys.path.insert(0,str(PROJECT_ROOT))


In [ ]:
import subprocess
subprocess.run([sys.executable,"-m","unittest","discover","-s","tests","-p","test_archive.py","-v"],cwd=PROJECT_ROOT,check=True)


## Tiny 합성 준비

분할 조각은 gzip의 중간에서 나뉘며 하나씩 독립 해제하지 않습니다. 결과와 준비 중 임시 파일은 `/content` 아래 source 프로젝트 outputs에 생성됩니다. Runtime이 삭제되면 소실되므로 필요한 결과·manifest는 별도 영구 보관합니다.


In [ ]:
import datetime, uuid, json
RUN_ROOT = PROJECT_ROOT/"outputs"/("archive_synthetic_"+datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S")+"_"+uuid.uuid4().hex[:8])
RUN_ROOT.mkdir(parents=True,exist_ok=False)
subprocess.run([sys.executable,"-m","rewind_archive.fixture","--output",str(RUN_ROOT/"fixture")],cwd=PROJECT_ROOT,check=True)
for kind in ("rgb","pose"):
    subprocess.run([sys.executable,"-m","rewind_archive","extract","--manifest",str(RUN_ROOT/"fixture"/f"{kind}_manifest.json"),
        "--limits",str(PROJECT_ROOT/"configs/archive_limits_smoke.json"),"--include","Fixture/Day/person/avatar/001/Motion__1_/*",
        "--output",str(RUN_ROOT/kind)],cwd=PROJECT_ROOT,check=True)
    subprocess.run([sys.executable,"-m","rewind_archive","verify","--extraction",str(RUN_ROOT/kind),
        "--manifest",str(RUN_ROOT/"fixture"/f"{kind}_manifest.json")],cwd=PROJECT_ROOT,check=True)
subprocess.run([sys.executable,"-m","rewind_archive","match","--extraction",str(RUN_ROOT/"rgb"),"--extraction",str(RUN_ROOT/"pose"),
    "--output",str(RUN_ROOT/"matching.json")],cwd=PROJECT_ROOT,check=True)
matching=json.loads((RUN_ROOT/"matching.json").read_text())
print({"matched_triplet_count":matching["matched_triplet_count"],"incomplete_frame_count":matching["incomplete_frame_count"],"train_ready":matching["train_ready"]})
assert matching["matched_triplet_count"]==4 and matching["train_ready"] is False


## 실제 Drive 준비 (별도 사용자 설정 후에만 실행)

**이 경로는 실제 Colab/Drive에서 아직 실행하지 않았습니다.** 아래 실행 여부를 True로 바꾸기 전, source ZIP과 별도로 준비한 `archive_manifest_v1`의 file ID·실제 size·version·modifiedTime과 연속 000-prefix 목록을 확인해야 합니다. Connector의 file 목록이나 크기만으로 version을 채우지 않습니다. metadata는 직접 인증된 Drive files.get으로 확인합니다. 인증 토큰을 manifest/code/log에 쓰지 않습니다.

다음 셀의 USER_MANIFEST_PATH와 SELECTED_MEMBER_GLOBS를 명시적으로 설정합니다. 이 notebook은 실제 원본의 자동 전체 다운로드·결합·추출을 하지 않습니다. CPU 순차 해제 후 얻은 완전한 파일도 관절·단위·좌표/calibration·timestamp·split/원본 motion/RGB 정합성 검증 전에는 학습하지 않습니다.

기본 상한은 `configs/archive_limits_smoke.json`에 있습니다. 설정 값과 보고서를 읽고 각 작업의 비용을 판단하세요. transfer_bytes는 어댑터가 받은 응답 본문(+Drive metadata)이며 HTTP/TLS 헤더·인증 교환·transport 선수신/재전송은 제외합니다. elapsed는 준비코어의 협력적 상한이고 DNS·auth·OS syscall 전체의 하드 deadline이 아닙니다. 프로세스 RAM 상한을 보장하지 않습니다.

Drive adapter는 Content-Length가 있는 200 metadata와 정확한 206 media, raw.read1 API가 필요합니다. 지원하지 않는 응답·API는 거부합니다. AuthorizedSession은 max_refresh_attempts=0 및 transport retry=0으로 구성합니다. 최초 auth와 credential refresh는 prepare 밖에서 수행하며 그 비용은 준비통계와 별도입니다. 준비 중 추가 refresh가 발생할 가능성도 있어 Colab에서 실제 지연·지원 API를 검증해야 합니다.


In [ ]:
RUN_DRIVE_PREPARATION = False
USER_MANIFEST_PATH = ""  # source ZIP 밖에 별도 업로드한 검증 manifest 경로
SELECTED_MEMBER_GLOBS = []  # 예: 정확한 sequence prefix + /json/*.json
if RUN_DRIVE_PREPARATION:
    if not USER_MANIFEST_PATH or not SELECTED_MEMBER_GLOBS:
        raise ValueError("원본 manifest와 선택할 sequence/member glob을 먼저 지정하세요.")
    from rewind_archive.__main__ import read_bounded_json
    from rewind_archive.contracts import Limits, validate_manifest
    from rewind_archive.extraction import prepare
    manifest_path=Path(USER_MANIFEST_PATH)
    manifest=validate_manifest(read_bounded_json(manifest_path))
    if manifest["source_scope"]!="user_drive" or any(p["source"]["type"]!="drive" for p in manifest["parts"]):
        raise ValueError("이 셀은 user_drive 계약과 drive source만 사용합니다.")
    limits=Limits.from_dict(read_bounded_json(PROJECT_ROOT/"configs/archive_limits_smoke.json"))
    print("limits:",json.dumps(limits.__dict__,indent=2))
    from google.colab import auth
    import google.auth
    from google.auth.transport.requests import AuthorizedSession, Request
    from requests.adapters import HTTPAdapter
    import urllib3
    if not callable(getattr(urllib3.response.HTTPResponse,"read1",None)):
        raise RuntimeError("urllib3 raw.read1 지원 API가 필요합니다. 현재 환경을 확인하세요.")
    print({"urllib3":urllib3.__version__,"live_drive_stack_verified":False})
    auth.authenticate_user()
    credentials,_=google.auth.default(scopes=["https://www.googleapis.com/auth/drive.readonly"])
    credentials.refresh(Request())  # token 값을 출력·저장하지 않는다.
    session=AuthorizedSession(credentials,max_refresh_attempts=0)
    session.mount("https://",HTTPAdapter(max_retries=0))
    destination=PROJECT_ROOT/"outputs"/("archive_drive_"+uuid.uuid4().hex)
    try:
        report=prepare(manifest,destination,SELECTED_MEMBER_GLOBS,limits,base=manifest_path.parent,authorized_session=session)
    finally:
        session.close()
    print({"output":str(destination),"stop_reason":report["stop_reason"],"stats":report["stats"],
        "inspection_candidates_usable":report["inspection_candidates_usable"],"train_ready":report["train_ready"]})
else:
    print("실제 Drive 인증/다운로드를 실행하지 않았습니다.")


## 재사용과 다음 단계

같은 런타임의 추출본은 verify로 manifest·개별 hash·code version을 검사해 재사용합니다. 이는 현재 원격 원본을 다시 확인하거나 전체 archive 무결성을 증명하지 않습니다. 손상 CRC·원본 변경·안전하지 않은 tar 때문에 unusable 상태인 추출본은 reuse/match가 거부됩니다. Runtime 재생성 시 추출본이 없으면 000부터 다시 읽는 비용이 발생합니다.

Matching의 valid는 이름 계약/중복 검사를 뜻하며 triplet 수는 별도입니다. 0 triplet이어도 문법이 맞으면 valid일 수 있습니다. Split은 unknown, 원본 motion은 unverified로 유지하고 모든 결과의 train_ready는 false입니다. 실제 데이터 검사와 RGB loader·M1 baseline 학습은 다음 지시 범위입니다.
